# ACT sul SO-101 — smistare caramelle

Impostazioni (gia' in `kernel-metadata.json`, lanciato con `kaggle kernels push`):
- **Accelerator: GPU T4 x2** (`machine_shape: NvidiaTeslaT4`). `lerobot-train`
  ne usa **una**: per due servirebbe `accelerate launch`, mai provato qui
- **Internet: On** (serve il telefono verificato sull'account)
- **Input**: il dataset privato in `dataset_sources` di `kernel-metadata.json`

Poi **Save Version → Save & Run All**: gira da solo, puoi chiudere il browser.
Il modello finito e' in **Output** → `modello_act.zip`.

## 1. La GPU c'e'?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Il dataset

Kaggle lo monta in sola lettura sotto `/kaggle/input/`: lo si cerca dal file
`meta/info.json` (non dipende dal nome scelto al caricamento) e lo si copia in
`/tmp`, fuori dall'output (che Kaggle salva e fa scaricare).

In [ ]:
import glob, json, shutil, os
trovati = glob.glob('/kaggle/input/**/meta/info.json', recursive=True)
assert trovati, 'Dataset non trovato: e\' aggiunto in Input?'
sorgente = os.path.dirname(os.path.dirname(trovati[0]))
shutil.copytree(sorgente, '/tmp/caramelle', dirs_exist_ok=True)
info = json.load(open('/tmp/caramelle/meta/info.json'))
print(sorgente, '->', '/tmp/caramelle')
print('episodi', info['total_episodes'], 'fotogrammi', info['total_frames'], 'fps', info['fps'])

## 3. LeRobot 0.6.1

Vuole Python >= 3.12, e Kaggle puo' averne uno piu' vecchio: `uv` crea un
ambiente 3.12 a parte in `/tmp` (fuori dall'output: torch pesa GB), con torch
per la GPU che trova (`--torch-backend=auto`). Stessa versione del portatile.

In [ ]:
!pip install -q uv
!UV_CACHE_DIR=/tmp/uvcache uv venv -q -p 3.12 /tmp/venv
!UV_CACHE_DIR=/tmp/uvcache uv pip install -q --python /tmp/venv/bin/python --torch-backend=auto "lerobot[training]==0.6.1"
!/tmp/venv/bin/python -c "import lerobot, torch; print('lerobot', lerobot.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 4. Addestramento

- `--steps=60000`: il minimo consigliato dagli autori di ACT per questi episodi
  (default LeRobot 100k)
- `--dataset.image_transforms.enable=true`: luminosita', contrasto, colore
  cambiati a caso, la rete non si lega alla luce della stanza
- `--policy.push_to_hub=false`: niente account Hugging Face
- ogni 200 passi stampa la `loss` (deve scendere) e i secondi per passo:
  da li' il tempo totale

In [ ]:
!/tmp/venv/bin/lerobot-train \
    --dataset.repo_id=so101/caramelle \
    --dataset.root=/tmp/caramelle \
    --dataset.image_transforms.enable=true \
    --policy.type=act \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --output_dir=/kaggle/working/act_caramelle \
    --job_name=act_caramelle \
    --steps=60000 \
    --save_freq=10000 \
    --batch_size=8 \
    --wandb.enable=false

## 5. I modelli da scaricare

Solo i pesi (`pretrained_model`) dei checkpoint a [30000, 60000] passi, uno zip
ciascuno (`modello_act_<passi>.zip`). Le cartelle si cercano per numero, non
per nome: LeRobot le numera con gli zeri davanti. Gli altri checkpoint restano
nell'output nel caso servano.

In [ ]:
import shutil, os
base = '/kaggle/working/act_caramelle/checkpoints'
numeri = {int(n): n for n in os.listdir(base) if n.isdigit()}
print('checkpoint:', sorted(numeri))
for passi in [30000, 60000]:
    cartella = os.path.join(base, numeri[passi], 'pretrained_model')
    uscita = shutil.make_archive(f'/kaggle/working/modello_act_{passi}', 'zip', cartella)
    print('pronto:', uscita, os.path.getsize(uscita) / 1e6, 'MB')